In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

GOOGLE_APPLICATION_CREDENTIALS = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
JAR_PATH = os.path.abspath("./lib/gcs-connector-hadoop3-2.2.5.jar")
SPARK_MASTER = 'spark://Adrians-M2-Macbook.local:7077'
# SPARK_MASTER = 'local[*]'

print(f"CREDENTIALS: [Exists: {os.path.exists(GOOGLE_APPLICATION_CREDENTIALS or '')}] {GOOGLE_APPLICATION_CREDENTIALS}")                                        
print(f"JAR: [Exists: {os.path.exists(JAR_PATH)}] {JAR_PATH}")                                                                 

CREDENTIALS: [Exists: True] /Users/acepeda/.config/gcloud/service_accounts/zoomcamp-sa.json
JAR: [Exists: True] /Users/acepeda/Documents/GitHub/zoomcamp-de/06-batch/code/lib/gcs-connector-hadoop3-2.2.5.jar


In [ ]:
# OLDER PATTERN TO CONNECT SPARK TO GCS
# conf = SparkConf() \
#     .setMaster(SPARK_MASTER) \
#     .setAppName('test') \
#     .set("spark.jars", JAR_PATH) \
#     .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
#     .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", GOOGLE_APPLICATION_CREDENTIALS)


# sc = SparkContext(conf=conf)

# hadoop_conf = sc._jsc.hadoopConfiguration()

# hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
# hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
# hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", GOOGLE_APPLICATION_CREDENTIALS)
# hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

In [ ]:
# EXPLICIT SPARK SESSION WITH HADOOP CONFIGURATION
# spark = SparkSession.builder \
#     .master(SPARK_MASTER) \
#     .config(conf=sc.getConf()) \
#     .getOrCreate()

In [2]:
# IMPLICIT SPARK SESSION WITH HADOOP CONFIGURATION
spark = SparkSession.builder \
    .master(SPARK_MASTER) \
    .appName('test') \
    .config("spark.jars", JAR_PATH) \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", GOOGLE_APPLICATION_CREDENTIALS) \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/18 11:41:21 WARN Utils: Your hostname, Adrians-M2-Macbook.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.97 instead (on interface en0)
26/03/18 11:41:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/18 11:41:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
df_green = spark.read.parquet('gs://dtc-de-course-488600-batch/green/*')

26/03/18 11:43:23 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://dtc-de-course-488600-batch/green/*.
java.io.FileNotFoundException: File not found: gs://dtc-de-course-488600-batch/green/*
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.lambda$getFileStatus$10(GoogleHadoopFileSystemBase.java:1085)
	at com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics.trackDuration(GhfsStorageStatistics.java:104)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getFileStatus(GoogleHadoopFileSystemBase.java:1070)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analys

In [6]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [10]:
import pyspark.sql.functions as F

df_green_revenue_daily = df_green\
    .withColumn('date', F.to_date('lpep_pickup_datetime'))\
    .select('date', 'total_amount')\
    .groupBy('date') \
    .agg(F.sum('total_amount').alias('revenue'))

In [11]:
df_green_revenue_daily.show(5)

+----------+------------------+
|      date|           revenue|
+----------+------------------+
|2019-02-23|  681480.559999819|
|2009-01-01|3487.1000000000013|
|2019-01-07| 649190.8999998296|
|2019-01-08| 656377.4599998337|
|2019-01-28| 681094.9399998238|
+----------+------------------+
only showing top 5 rows


26/03/18 12:35:18 ERROR TaskSchedulerImpl: Lost executor 0 on 10.0.0.97: worker lost: Not receiving heartbeat for 60 seconds
26/03/18 14:39:21 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 1898432 ms exceeds timeout 120000 ms
26/03/18 14:39:21 ERROR TaskSchedulerImpl: Lost executor 1 on 10.0.0.97: Executor heartbeat timed out after 1898432 ms
26/03/19 06:34:23 WARN HeartbeatReceiver: Removing executor 2 with no recent heartbeats: 276347 ms exceeds timeout 120000 ms
26/03/19 06:34:23 ERROR TaskSchedulerImpl: Lost executor 2 on 10.0.0.97: Executor heartbeat timed out after 276347 ms
26/03/19 07:12:34 WARN HeartbeatReceiver: Removing executor 3 with no recent heartbeats: 151370 ms exceeds timeout 120000 ms
26/03/19 07:12:34 ERROR TaskSchedulerImpl: Lost executor 3 on 10.0.0.97: Executor heartbeat timed out after 151370 ms
26/03/19 07:28:21 ERROR TaskSchedulerImpl: Lost executor 4 on 10.0.0.97: worker lost: Not receiving heartbeat for 60 seconds
26/03/20 07:57:14 W